# Saber BERM Scope v2 — 全量 GSM8K 验证

基于 v1 (LIMIT=150) 的结论，trail_block 是最有前途的方向。
本轮在 trail_block 的 n=2~8, mu=4~16 做细粒度扫描（30 tasks, LIMIT=150）。

分析时会合并 v1 的 24 个结果一起看。

## 1. 环境设置

In [ ]:
import os, gc, re, json, datetime, threading, queue, subprocess
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '4,5')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)
os.makedirs('evals_results/saber', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

## 2. 任务配置（30 tasks, LIMIT=150）

In [ ]:
task = 'gsm8k'
fewshot = 5
seed = 42
gen_length = 256
steps = 256
limit_samples = 150
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


def _task(name, n, mu, berm_scope='window'):
    extra_args = [
        'saber_expand=True',
        'berm_mode=cross_step',
        'saber_global_aadu=True',
        'saber_mtr=0.8',
        f'saber_n={n}',
        f'saber_mu={mu}',
        'block_length=32',
        f'saber_berm_scope={berm_scope}',
    ]
    return {'name': name, 'extra_args': extra_args}


TASK_CONFIGS = [
    # anchor
    _task('v2_anchor_n4_mu8', 4, 8, 'window'),

    # ── n=3 完整 mu 扫描 ──
    _task('v2_trail_n3_mu4', 3, 4, 'trail_block'),
    _task('v2_trail_n3_mu6', 3, 6, 'trail_block'),
    _task('v2_trail_n3_mu8', 3, 8, 'trail_block'),
    _task('v2_trail_n3_mu10', 3, 10, 'trail_block'),
    _task('v2_trail_n3_mu12', 3, 12, 'trail_block'),
    _task('v2_trail_n3_mu16', 3, 16, 'trail_block'),

    # ── n=4 完整 mu 扫描 ──
    _task('v2_trail_n4_mu4', 4, 4, 'trail_block'),
    _task('v2_trail_n4_mu6', 4, 6, 'trail_block'),
    _task('v2_trail_n4_mu8', 4, 8, 'trail_block'),
    _task('v2_trail_n4_mu10', 4, 10, 'trail_block'),
    _task('v2_trail_n4_mu12', 4, 12, 'trail_block'),
    _task('v2_trail_n4_mu16', 4, 16, 'trail_block'),

    # ── n=5 完整 mu 扫描 ──
    _task('v2_trail_n5_mu4', 5, 4, 'trail_block'),
    _task('v2_trail_n5_mu6', 5, 6, 'trail_block'),
    _task('v2_trail_n5_mu8', 5, 8, 'trail_block'),
    _task('v2_trail_n5_mu10', 5, 10, 'trail_block'),
    _task('v2_trail_n5_mu12', 5, 12, 'trail_block'),
    _task('v2_trail_n5_mu16', 5, 16, 'trail_block'),

    # ── n=6 mu 扫描 ──
    _task('v2_trail_n6_mu4', 6, 4, 'trail_block'),
    _task('v2_trail_n6_mu8', 6, 8, 'trail_block'),
    _task('v2_trail_n6_mu12', 6, 12, 'trail_block'),
    _task('v2_trail_n6_mu16', 6, 16, 'trail_block'),

    # ── n=7 关键点 ──
    _task('v2_trail_n7_mu8', 7, 8, 'trail_block'),
    _task('v2_trail_n7_mu12', 7, 12, 'trail_block'),

    # ── n=2 极保守 ──
    _task('v2_trail_n2_mu8', 2, 8, 'trail_block'),
    _task('v2_trail_n2_mu12', 2, 12, 'trail_block'),

    # ── n=8 极激进 ──
    _task('v2_trail_n8_mu8', 8, 8, 'trail_block'),
]

GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
print(f'Total tasks: {len(TASK_CONFIGS)} | GPUs: {GPU_POOL} | FULL GSM8K | timestamp: {timestamp}')

## 3. 并行启动任务

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['name']
        log_file = f'nlogs/sweep_bermscope_v2_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

        common_args = [
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            'show_speed=True',
            f'seed={seed}',
        ]
        model_args = ','.join(common_args + cfg['extra_args'])
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot} --confirm_run_unsafe_code '
            f'--model llada_dist --model_args {model_args} --output_path {output_dir} --log_samples '
            f'--limit {limit_samples}'
        )

        print(f'[GPU {gpu_id}] START {name}')
        p = subprocess.Popen(
            cmd,
            shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()
        print(f'[GPU {gpu_id}] DONE  {name} rc={rc}')
        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))
        task_queue.task_done()


threads = []
for gid in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gid,), daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print('All finished:', len(all_results), '/', len(TASK_CONFIGS))

## 4. 解析 v2 结果

In [ ]:
def _parse_one(name, output_dir, log_file):
    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    log_content = Path(log_file).read_text(encoding='utf-8', errors='ignore') if Path(log_file).exists() else ''

    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)

    scope_m = re.search(r'(?:bs_|v2_)(\w+?)_n', name)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)

    return {
        'name': name,
        'scope': scope_m.group(1) if scope_m else '?',
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    }


rows = []
for cfg, name, log_file, output_dir, rc in all_results:
    rows.append(_parse_one(name, output_dir, log_file))

df_v2 = pd.DataFrame(rows).sort_values(['scope', 'n', 'mu'])
pd.set_option('display.max_rows', 50)
display(df_v2)

## 5. 合并 v1 结果 + 综合分析

自动加载 v1 (LIMIT=150) 的 24 个结果，和 v2 放在一起对比。
注意 v1 是 150 样本，v2 是全量 1319 样本，FlexAcc 数值不可直接比，但 NFE/速度可比。

In [ ]:
# Load v1 results
import glob

v1_names = [
    'saber_exp_gl_cs_n4_mu8_bl32_base',
    'bs_curblk_n3_mu8', 'bs_curblk_n4_mu8', 'bs_curblk_n5_mu8',
    'bs_curblk_n6_mu8', 'bs_curblk_n8_mu8',
    'bs_curblk_n4_mu4', 'bs_curblk_n4_mu6', 'bs_curblk_n4_mu12',
    'bs_ffberm_n4_mu8', 'bs_ffberm_n5_mu8', 'bs_ffberm_n6_mu8',
    'bs_ffberm_n4_mu4', 'bs_ffberm_n4_mu12',
    'bs_trail_n3_mu8', 'bs_trail_n4_mu8', 'bs_trail_n5_mu8',
    'bs_trail_n6_mu8', 'bs_trail_n8_mu8',
    'bs_trail_n4_mu4', 'bs_trail_n4_mu6', 'bs_trail_n4_mu12',
    'bs_trail_n6_mu4', 'bs_trail_n6_mu12',
]

# Auto-detect v1 timestamp
v1_logs = sorted(
    glob.glob(f'nlogs/sweep_bermscope_{task}_saber_exp_gl_cs_n4_mu8_bl32_base_*.log'),
    key=os.path.getmtime, reverse=True,
)
v1_ts = None
if v1_logs:
    v1_ts = os.path.basename(v1_logs[0]).replace(f'sweep_bermscope_{task}_saber_exp_gl_cs_n4_mu8_bl32_base_', '').replace('.log', '')
    print(f'v1 timestamp: {v1_ts}')
else:
    print('WARNING: v1 logs not found')

v1_rows = []
if v1_ts:
    for name in v1_names:
        lf = f'nlogs/sweep_bermscope_{task}_{name}_{v1_ts}.log'
        od = f'evals_results/saber/{task}-{name}-{v1_ts}'
        if os.path.exists(lf):
            r = _parse_one(name, od, lf)
            r['source'] = 'v1_150'
            v1_rows.append(r)

# Tag v2
for r in rows:
    r['source'] = 'v2_full'

df_all = pd.DataFrame(v1_rows + rows).sort_values(['scope', 'n', 'mu', 'source'])
pd.set_option('display.max_rows', 100)

print(f'\nCombined: {len(v1_rows)} v1 + {len(rows)} v2 = {len(v1_rows)+len(rows)} total')
print(f'\n{"Name":<36} {"Src":<8} {"Scope":<10} {"n":<4} {"mu":<5} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10}')
print('-' * 104)
for _, r in df_all.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    src = r.get('source', '?')
    print(f"{r['name']:<36} {src:<8} {r['scope']:<10} {r['n']:<4} {r['mu']:<5} {fa:<10} {sa:<11} {sp:<10} {nf:<10}")

## 6. 对比图（仅 v2 全量结果）

In [ ]:
plot_df = df_v2[df_v2['flex_acc'].notna()].copy()
plot_df['label'] = plot_df['name']

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
metrics = [('flex_acc', 'FlexAcc'), ('total_nfe', 'Total NFE'), ('tok_per_sec', 'Tokens/sec')]

scope_colors = {'anchor': '#E53935', 'curblk': '#4E79A7', 'trail': '#59A14F'}
colors = [scope_colors.get(r['scope'], '#999') for _, r in plot_df.iterrows()]

for ax, (col, title) in zip(axes, metrics):
    vals = plot_df[col].fillna(0)
    ax.bar(plot_df['label'], vals, color=colors)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[-1].tick_params(axis='x', rotation=55, labelsize=8)
plt.tight_layout()
plt.show()

## 7. 从已有结果重新加载（可选）

内核重启后运行此 cell + Section 5/6。

In [ ]:
def _parse_one(name, output_dir, log_file):
    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    log_content = Path(log_file).read_text(encoding='utf-8', errors='ignore') if Path(log_file).exists() else ''

    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)

    scope_m = re.search(r'(?:bs_|v2_)(\w+?)_n', name)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)

    return {
        'name': name,
        'scope': scope_m.group(1) if scope_m else '?',
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    }


import glob

task = 'gsm8k'
seed = 42

V2_NAMES = [
    'v2_anchor_n4_mu8',
    'v2_trail_n2_mu8', 'v2_trail_n2_mu12',
    'v2_trail_n3_mu4', 'v2_trail_n3_mu6', 'v2_trail_n3_mu8',
    'v2_trail_n3_mu10', 'v2_trail_n3_mu12', 'v2_trail_n3_mu16',
    'v2_trail_n4_mu4', 'v2_trail_n4_mu6', 'v2_trail_n4_mu8',
    'v2_trail_n4_mu10', 'v2_trail_n4_mu12', 'v2_trail_n4_mu16',
    'v2_trail_n5_mu4', 'v2_trail_n5_mu6', 'v2_trail_n5_mu8',
    'v2_trail_n5_mu10', 'v2_trail_n5_mu12', 'v2_trail_n5_mu16',
    'v2_trail_n6_mu4', 'v2_trail_n6_mu8', 'v2_trail_n6_mu12', 'v2_trail_n6_mu16',
    'v2_trail_n7_mu8', 'v2_trail_n7_mu12',
    'v2_trail_n8_mu8',
]

latest = sorted(
    glob.glob(f'nlogs/sweep_bermscope_v2_{task}_v2_anchor_n4_mu8_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest:
    fname = os.path.basename(latest[0])
    timestamp = fname.replace(f'sweep_bermscope_v2_{task}_v2_anchor_n4_mu8_', '').replace('.log', '')
    print(f'Auto-detected v2 timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No v2 log found!')

rows = []
for name in V2_NAMES:
    lf = f'nlogs/sweep_bermscope_v2_{task}_{name}_{timestamp}.log'
    od = f'evals_results/saber/{task}-{name}-{timestamp}'
    if not os.path.exists(lf):
        print(f'  [MISS] {name}')
        continue
    rows.append(_parse_one(name, od, lf))
    print(f'  [OK]   {name}')

df_v2 = pd.DataFrame(rows).sort_values(['scope', 'n', 'mu'])
df = df_v2

print(f'\nLoaded {len(rows)} v2 results (timestamp={timestamp})')
print(f'\n{"Name":<28} {"Scope":<10} {"n":<4} {"mu":<5} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 96)
for _, r in df_v2.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<28} {r['scope']:<10} {r['n']:<4} {r['mu']:<5} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

print(f'\nRe-run Section 5/6 for combined analysis and plots.')